# 01: Data Download

### Data Download for Clinical, Microenvironment, and Expression data
1. Go to https://portal.gdc.cancer.gov/analysis_page?app=CohortBuilder&tab=general
    - Select Program: TCGA
    - "# CASES" -> "Table View" -> "TSV"
    - Download to `data_raw/`
2. Go to "Repository" with the previous TCGA selection still active.
2. Make the following selections: 
    - Experimental Strategy: RNA-Seq
    - Access: open
    - Tumor Descriptor: primary, not applicable
    - Specimen Type: solid tissue
    - Preservation Method: unknown
3. "Add All Files to Cart". Go to "Cart". 
4. Place the following downloads in the `data_raw/unknown` folder.
    - Download Associated Data
        - Clinical: TSV
        - Biospecimen: TSV
        - Sample Sheet
    - Download Cart
        - Manifest
    - Unzip these files
4. "Remove from Cart" -> "All Files"
4. Go back to downloads, select the same filters but change
    - Preservation Method: all except unknown (should only be oct and ffpe)
5. Repeat the download, placing in `data_raw/other`.
6. Use the GDC client CLI to download rnaseq files from both manifests (~60GB)
```
cd data_raw/unknown/rnaseq
gdc-client download -m ../<your_unknown_manifest>.txt
cd ../../other/rnaseq
gdc-client download -m ../<your_other_manifest>.txt
```

### Data Download for SNV and SCNA data
This data is under access controls. Please refer to the cited studies in our supplementary material for compiling this data.

In [1]:
import os
import pandas as pd
import numpy as np

# Replace these paths to run for both the "unknown" and "other" downloads
in_dir = 'data_raw/unknown'
out_dir = 'data_intermediate/unknown'
os.makedirs(out_dir, exist_ok=True)

# Replace these with your folder names
tcga_cases_tsv = 'data_raw/cases.tsv'
tcga_clinical_tsv = os.path.join(in_dir, 'clinical.cart.2025-01-17/clinical.tsv')
tcga_sample_tsv = os.path.join(in_dir, 'biospecimen.cart.2025-01-17/sample.tsv')
tcga_slide_tsv = os.path.join(in_dir, 'biospecimen.cart.2025-01-17/slide.tsv')
tcga_file_tsv = os.path.join(in_dir, 'gdc_sample_sheet.2025-01-17.tsv')

In [2]:
tcga_cases_df = pd.read_csv(tcga_cases_tsv, sep='\t')
tcga_clinical_df = pd.read_csv(tcga_clinical_tsv, sep='\t')
tcga_sample_df = pd.read_csv(tcga_sample_tsv, sep='\t')
tcga_slide_df = pd.read_csv(tcga_slide_tsv, sep='\t')
tcga_file_df = pd.read_csv(tcga_file_tsv, sep='\t')

/var/folders/_6/468mk1cx3cb080z2r5gq0k_r0000gn/T/ipykernel_4298/206510135.py:1: DtypeWarning: Columns (11,13,15,17,19,21,23,25,27,29,33,35,37,39,41,43,45,47,61) have mixed types. Specify dtype option on import or set low_memory=False.
  tcga_cases_df = pd.read_csv(tcga_cases_tsv, sep='\t')


In [3]:
selected_clinical_cols = [
    'case_id',
    'case_submitter_id',
    'project_id',
    'age_at_diagnosis',
    'gender',
    'year_of_birth',
    'race',
    'ajcc_pathologic_stage',
    'ajcc_clinical_stage',
    'ann_arbor_clinical_stage',
    'figo_stage',
]
selected_survival_cols = [
    'case_id',
    'case_submitter_id',
    'vital_status',
    'days_to_death',
    'days_to_last_follow_up',
]
selected_sample_cols = [
    'case_id',
    'case_submitter_id',
    'sample_id',
    'sample_submitter_id',
    'days_to_collection',
    'sample_type',
]
selected_slide_cols = [
    'case_id',
    'case_submitter_id',
    'sample_id',
    'sample_submitter_id',
    'slide_submitter_id',
    'percent_neutrophil_infiltration',
    'percent_monocyte_infiltration',
    'percent_normal_cells',
    'percent_tumor_nuclei',
    'percent_lymphocyte_infiltration',
    'percent_stromal_cells',
    'percent_tumor_cells',
]
disease_names = {
    "LAML": "Acute Myeloid Leukemia",
    "ACC": "Adrenocortical carcinoma",
    "BLCA": "Bladder Urothelial Carcinoma",
    "LGG": "Brain Lower Grade Glioma",
    "BRCA": "Breast invasive carcinoma",
    "CESC": "Cervical squamous cell carcinoma and endocervical adenocarcinoma",
    "CHOL": "Cholangiocarcinoma",
    "LCML": "Chronic Myelogenous Leukemia",
    "COAD": "Colon adenocarcinoma",
    "ESCA": "Esophageal carcinoma",
    "GBM": "Glioblastoma multiforme",
    "HNSC": "Head and Neck squamous cell carcinoma",
    "KICH": "Kidney Chromophobe",
    "KIRC": "Kidney renal clear cell carcinoma",
    "KIRP": "Kidney renal papillary cell carcinoma",
    "LIHC": "Liver hepatocellular carcinoma",
    "LUAD": "Lung adenocarcinoma",
    "LUSC": "Lung squamous cell carcinoma",
    "DLBC": "Lymphoid Neoplasm Diffuse Large B-cell Lymphoma",
    "MESO": "Mesothelioma",
    "MISC": "Miscellaneous",
    "OV": "Ovarian serous cystadenocarcinoma",
    "PAAD": "Pancreatic adenocarcinoma",
    "PCPG": "Pheochromocytoma and Paraganglioma",
    "PRAD": "Prostate adenocarcinoma",
    "READ": "Rectum adenocarcinoma",
    "SARC": "Sarcoma",
    "SKCM": "Skin Cutaneous Melanoma",
    "STAD": "Stomach adenocarcinoma",
    "TGCT": "Testicular Germ Cell Tumors",
    "THYM": "Thymoma",
    "THCA": "Thyroid carcinoma",
    "UCS": "Uterine Carcinosarcoma",
    "UCEC": "Uterine Corpus Endometrial Carcinoma",
    "UVM": "Uveal Melanoma",
}
primary_sites = {
    'LAML': 'Blood',
    'ACC': 'Adrenal Gland',
    'BLCA': 'Bladder',
    'LGG': 'Brain',
    'BRCA': 'Breast',
    'CESC': 'Cervix',
    'CHOL': 'Bile Duct',
    'LCML': 'Blood',
    'COAD': 'Colorectal',
    'ESCA': 'Esophagus',
    'GBM': 'Brain',
    'HNSC': 'Head and Neck',
    'KICH': 'Kidney',
    'KIRC': 'Kidney',
    'KIRP': 'Kidney',
    'LIHC': 'Liver',
    'LUAD': 'Lung',
    'LUSC': 'Lung',
    'DLBC': 'Lymph Nodes',
    'MESO': 'Pleura',
    'MISC': 'Miscellanteous',
    'OV': 'Ovary',
    'PAAD': 'Pancreas',
    'PCPG': 'Adrenal Gland',
    'PRAD': 'Prostate',
    'READ': 'Colorectal',
    'SARC': 'Soft Tissue',
    'SKCM': 'Skin',
    'STAD': 'Stomach',
    'TGCT': 'Testis',
    'THYM': 'Thymus',
    'THCA': 'Thyroid',
    'UCS': 'Uterus',
    'UCEC': 'Uterus',
    'UVM': 'Eye'
}

In [4]:
print('SURVIVAL')
tcga_survival_df = tcga_clinical_df[selected_survival_cols]
display(tcga_survival_df.head())
# print('CASES')
# tcga_cases_df = tcga_cases_df[selected_case_cols]
# display(tcga_cases_df.head())
print('CLINICAL')
tcga_clinical_df = tcga_clinical_df[selected_clinical_cols]
display(tcga_clinical_df.head())
print('SAMPLE')
tcga_sample_df = tcga_sample_df[selected_sample_cols]
display(tcga_sample_df.head())
print('SLIDE')
tcga_slide_df = tcga_slide_df[selected_slide_cols]
display(tcga_slide_df.head())
print('RNASEQ FILES')
tcga_file_df.drop_duplicates(subset='Sample ID', inplace=True, keep='last')
display(tcga_file_df.head())

SURVIVAL


,case_id,case_submitter_id,vital_status,days_to_death,days_to_last_follow_up
0,0011a67b-1ba9-4a32-a6b8-7850759a38cf,TCGA-DC-6158,Dead,334,216.0
1,0011a67b-1ba9-4a32-a6b8-7850759a38cf,TCGA-DC-6158,Dead,334,216.0
2,001944e5-af34-4061-9c09-bb9ea346f6fd,TCGA-HQ-A5ND,Dead,274,'--
3,001944e5-af34-4061-9c09-bb9ea346f6fd,TCGA-HQ-A5ND,Dead,274,'--
4,001ad307-4ad3-4f1d-b2fc-efc032871c7e,TCGA-HT-A614,Alive,'--,82.0


CLINICAL


,case_id,case_submitter_id,project_id,age_at_diagnosis,gender,year_of_birth,race,ajcc_pathologic_stage,ajcc_clinical_stage,ann_arbor_clinical_stage,figo_stage
0,0011a67b-1ba9-4a32-a6b8-7850759a38cf,TCGA-DC-6158,TCGA-READ,25842,male,1940,white,Stage I,'--,'--,'--
1,0011a67b-1ba9-4a32-a6b8-7850759a38cf,TCGA-DC-6158,TCGA-READ,25842,male,1940,white,Stage I,'--,'--,'--
2,001944e5-af34-4061-9c09-bb9ea346f6fd,TCGA-HQ-A5ND,TCGA-BLCA,28555,male,'--,not reported,Stage IV,'--,'--,'--
3,001944e5-af34-4061-9c09-bb9ea346f6fd,TCGA-HQ-A5ND,TCGA-BLCA,28555,male,'--,not reported,Stage IV,'--,'--,'--
4,001ad307-4ad3-4f1d-b2fc-efc032871c7e,TCGA-HT-A614,TCGA-LGG,17392,male,1966,white,'--,'--,'--,'--


SAMPLE


,case_id,case_submitter_id,sample_id,sample_submitter_id,days_to_collection,sample_type
0,01ad5016-f691-4bca-82a0-910429d8d25b,TCGA-AA-3561,5e797e09-26d0-42f4-838f-3b05e489e6ec,TCGA-AA-3561-01A,'--,Primary Tumor
1,01ad5016-f691-4bca-82a0-910429d8d25b,TCGA-AA-3561,699f2743-46f7-4967-b670-30c93a59ab0f,TCGA-AA-3561-01Z,'--,Primary Tumor
2,01ad5016-f691-4bca-82a0-910429d8d25b,TCGA-AA-3561,69cff594-84a3-4d68-9edd-025b7d540535,TCGA-AA-3561-10A,'--,Blood Derived Normal
3,01ad5016-f691-4bca-82a0-910429d8d25b,TCGA-AA-3561,eb29b3dc-6291-490c-b143-cd7c235904e2,TCGA-AA-3561-10B,'--,Blood Derived Normal
4,77200abe-4a09-4a4a-bb45-60d9b96d4998,TCGA-EL-A3H2,405fbe40-5934-431a-8806-325ac0d788ed,TCGA-EL-A3H2-11A,3292,Solid Tissue Normal


SLIDE


,case_id,case_submitter_id,sample_id,sample_submitter_id,slide_submitter_id,percent_neutrophil_infiltration,percent_monocyte_infiltration,percent_normal_cells,percent_tumor_nuclei,percent_lymphocyte_infiltration,percent_stromal_cells,percent_tumor_cells
0,01ad5016-f691-4bca-82a0-910429d8d25b,TCGA-AA-3561,5e797e09-26d0-42f4-838f-3b05e489e6ec,TCGA-AA-3561-01A,TCGA-AA-3561-01A-01-BS1,'--,'--,0.0,90.0,'--,10.0,90.0
1,01ad5016-f691-4bca-82a0-910429d8d25b,TCGA-AA-3561,5e797e09-26d0-42f4-838f-3b05e489e6ec,TCGA-AA-3561-01A,TCGA-AA-3561-01A-01-TS1,'--,'--,0.0,90.0,'--,20.0,80.0
2,01ad5016-f691-4bca-82a0-910429d8d25b,TCGA-AA-3561,699f2743-46f7-4967-b670-30c93a59ab0f,TCGA-AA-3561-01Z,TCGA-AA-3561-01Z-00-DX1,'--,'--,'--,'--,'--,'--,'--
3,77200abe-4a09-4a4a-bb45-60d9b96d4998,TCGA-EL-A3H2,405fbe40-5934-431a-8806-325ac0d788ed,TCGA-EL-A3H2-11A,TCGA-EL-A3H2-11A-01-TS1,0.0,5.0,100.0,0.0,2.0,0.0,0.0
4,77200abe-4a09-4a4a-bb45-60d9b96d4998,TCGA-EL-A3H2,64e11d42-fec9-47e4-a6a6-8df2a6bfd79b,TCGA-EL-A3H2-01A,TCGA-EL-A3H2-01A-01-TS1,0.0,0.0,0.0,95.0,0.0,5.0,95.0


RNASEQ FILES


,File ID,File Name,Data Category,Data Type,Project ID,Case ID,Sample ID,Sample Type
0,98e7190a-4d3a-4fdb-a021-c482f87e65f2,da658a99-8972-473e-9754-9712be0b1b25.rna_seq.a...,Transcriptome Profiling,Gene Expression Quantification,TCGA-ESCA,TCGA-R6-A6XQ,TCGA-R6-A6XQ-01B,Primary Tumor
1,3da6be11-66bd-437b-9ce6-87e63beedbec,20f5e7d5-0faf-47f6-9e7c-ecef50a58288.rna_seq.a...,Transcriptome Profiling,Gene Expression Quantification,TCGA-ESCA,TCGA-R6-A6KZ,TCGA-R6-A6KZ-01A,Primary Tumor
2,79ce65af-7af4-45bd-93ea-ce87652a1f8d,25961c5e-75a1-4f22-b3b8-6f373dad4d98.rna_seq.a...,Transcriptome Profiling,Gene Expression Quantification,TCGA-ESCA,TCGA-L5-A43E,TCGA-L5-A43E-01A,Primary Tumor
3,5d321eee-f8c4-4460-b4ee-4a3f016206e4,93d85034-6192-4ee1-a0ec-8126e8c049a4.rna_seq.a...,Transcriptome Profiling,Gene Expression Quantification,TCGA-ESCA,TCGA-JY-A6F8,TCGA-JY-A6F8-01A,Primary Tumor
4,c702ce91-1d1d-4d1c-ba9b-1dbe937425e2,a3e18366-19fa-4d0a-96e8-f15af6393a87.rna_seq.a...,Transcriptome Profiling,Gene Expression Quantification,TCGA-ESCA,TCGA-RE-A7BO,TCGA-RE-A7BO-01A,Primary Tumor


In [5]:
# Compile clinical covariates
my_clinical_df = tcga_clinical_df.copy()
my_clinical_df['disease_type'] = tcga_clinical_df['project_id'].map(lambda s: s.split('-')[1])
my_clinical_df.drop(columns=['project_id', 'case_submitter_id'], inplace=True)
my_clinical_df['disease_name'] = my_clinical_df['disease_type'].map(disease_names)
my_clinical_df['primary_site'] = my_clinical_df['disease_type'].map(primary_sites)
my_clinical_df.drop_duplicates(inplace=True)
my_clinical_df.replace("'--", None, inplace=True)

stage_features = [
    'ajcc_pathologic_stage',
    'ajcc_clinical_stage',
    'ann_arbor_clinical_stage',
    'figo_stage',
]
stages = []
for i, (ajcc_path, ajcc_clin, ann, figo) in my_clinical_df[stage_features].iterrows():
    def get_stage(s):
        if "IV" in s:
            return 4.
        elif "III" in s:
            return 3.
        elif "II" in s:
            return 2.
        elif "I" in s:
            return 1.
        else:
            return None
    if ajcc_path is not None:
        stage = get_stage(ajcc_path)
    elif ajcc_clin is not None:
        stage = get_stage(ajcc_clin)
    elif ann is not None:
        stage = get_stage(ann)
    elif figo is not None:
        stage = get_stage(figo)
    else:
        stage = None
    stages.append(stage)
my_clinical_df['stage'] = stages 
my_clinical_df.drop(columns=stage_features, inplace=True)

my_clinical_df.head()

,case_id,age_at_diagnosis,gender,year_of_birth,race,disease_type,disease_name,primary_site,stage
0,0011a67b-1ba9-4a32-a6b8-7850759a38cf,25842,male,1940,white,READ,Rectum adenocarcinoma,Colorectal,1.0
2,001944e5-af34-4061-9c09-bb9ea346f6fd,28555,male,None,not reported,BLCA,Bladder Urothelial Carcinoma,Bladder,4.0
4,001ad307-4ad3-4f1d-b2fc-efc032871c7e,17392,male,1966,white,LGG,Brain Lower Grade Glioma,Brain,NaN
6,001cef41-ff86-4d3f-a140-a647ac4b10a1,22279,female,1950,white,BRCA,Breast invasive carcinoma,Breast,1.0
8,001e0309-9c50-42b0-9e38-347883ee2cd3,17045,female,1963,not reported,UCEC,Uterine Corpus Endometrial Carcinoma,Uterus,1.0


In [6]:
# Compile sample covariates and merge with clinical covariates
# Only use the first tumor and healthy sample from each patient
my_sample_df = tcga_sample_df[tcga_sample_df['sample_submitter_id'].map(lambda s: '-11A' in s or '-01A' in s)]
my_sample_df = my_sample_df.merge(tcga_slide_df.drop(columns=['case_id', 'case_submitter_id', 'sample_submitter_id', 'slide_submitter_id']), on='sample_id')
my_sample_df.drop_duplicates(subset='sample_id', inplace=True)  # Multiple slides, just take the first
my_sample_df.replace("'--", None, inplace=True)

# Merge patient-level clinical covariates
my_sample_df = my_sample_df.merge(my_clinical_df, on='case_id', how='left')
my_sample_df['sample_id'] = my_sample_df['sample_submitter_id']
my_sample_df['submitter_id'] = my_sample_df['case_submitter_id']
my_sample_df.drop(columns=['case_id', 'case_submitter_id', 'sample_submitter_id'], inplace=True)

my_sample_df.to_csv(os.path.join(out_dir, 'clinical_covariates.csv'), index=False)
my_sample_df.head()

,sample_id,days_to_collection,sample_type,percent_neutrophil_infiltration,percent_monocyte_infiltration,percent_normal_cells,percent_tumor_nuclei,percent_lymphocyte_infiltration,percent_stromal_cells,percent_tumor_cells,age_at_diagnosis,gender,year_of_birth,race,disease_type,disease_name,primary_site,stage,submitter_id
0,TCGA-AA-3561-01A,None,Primary Tumor,None,None,0.0,90.0,None,10.0,90.0,26420,male,1937,not reported,COAD,Colon adenocarcinoma,Colorectal,2.0,TCGA-AA-3561
1,TCGA-EL-A3H2-11A,3292,Solid Tissue Normal,0.0,5.0,100.0,0.0,2.0,0.0,0.0,21266,male,1944,white,THCA,Thyroid carcinoma,Thyroid,2.0,TCGA-EL-A3H2
2,TCGA-EL-A3H2-01A,3292,Primary Tumor,0.0,0.0,0.0,95.0,0.0,5.0,95.0,21266,male,1944,white,THCA,Thyroid carcinoma,Thyroid,2.0,TCGA-EL-A3H2
3,TCGA-L5-A88S-01A,155,Primary Tumor,0.0,0.0,30.0,75.0,2.0,0.0,70.0,30954,male,1929,white,ESCA,Esophageal carcinoma,Esophagus,1.0,TCGA-L5-A88S
4,TCGA-L5-A88S-11A,155,Solid Tissue Normal,0.0,0.0,100.0,0.0,4.0,0.0,0.0,30954,male,1929,white,ESCA,Esophageal carcinoma,Esophagus,1.0,TCGA-L5-A88S


In [7]:
# Survival
my_survival_df = tcga_survival_df[['case_submitter_id', 'vital_status', 'days_to_death', 'days_to_last_follow_up']]
my_survival_df['died'] = my_survival_df['vital_status'] == 'Dead'
my_survival_df['time'] = my_survival_df['days_to_death']
my_survival_df.replace("'--", None, inplace=True)
my_survival_df['time'].fillna(my_survival_df['days_to_last_follow_up'], inplace=True)
my_survival_df.drop(columns=['vital_status', 'days_to_death', 'days_to_last_follow_up'], inplace=True)
my_survival_df.drop_duplicates(inplace=True)
my_survival_df.rename({'case_submitter_id': 'submitter_id'}, axis=1, inplace=True)

my_survival_df.to_csv(os.path.join(out_dir, 'survival.csv'), index=False)
my_survival_df.head()

/var/folders/_6/468mk1cx3cb080z2r5gq0k_r0000gn/T/ipykernel_4298/925774716.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  my_survival_df['time'].fillna(my_survival_df['days_to_last_follow_up'], inplace=True)


,submitter_id,died,time
0,TCGA-DC-6158,True,334
2,TCGA-HQ-A5ND,True,274
4,TCGA-HT-A614,False,82.0
6,TCGA-E2-A1IU,False,337.0
8,TCGA-D1-A17N,False,46.0


In [8]:
# Gene expression
gene_names = np.loadtxt('data/cancer_genes.txt', dtype=str)

def read_rnaseq(sample_id, count_type = 'fpkm_uq_unstranded'):
    # count_type = unstranded, stranded_first, stranded_second, tpm_unstranded, fpkm_unstranded, fpkm_uq_unstranded
    sample_file_df = tcga_file_df[tcga_file_df['Sample ID'] == sample_id]
    if len(sample_file_df) < 1:
        return
    if len(sample_file_df) > 1:
        display(sample_file_df)
        raise ValueError(f"{sample_id} has multiple files")
    file_id = sample_file_df['File ID'].item()
    file_name = sample_file_df['File Name'].item()
    filepath = os.path.join(in_dir, 'rnaseq', file_id, file_name)
    df = pd.read_csv(filepath, sep='\t', header=1)
    df = df.drop_duplicates(subset='gene_name', keep='last')
    df = df[df['gene_name'].isin(gene_names)]
    df = df[['gene_name', count_type]]
    df[count_type] = 1e4 / df[count_type].sum() * df[count_type]
    df[count_type] = np.log10(df[count_type] + 1e-3)
    df.index = df['gene_name'].values
    df = df.drop(columns=['gene_name'])
    df = df.T.reset_index()
    df = df.drop(columns=['index'])
    df['sample_id'] = sample_id
    df = df[['sample_id'] + df.columns.tolist()[:-1]]
    return df

expression_rows = []
for i, sample_id in my_sample_df['sample_id'].items():
    print(len(my_sample_df), i, end='\r')
    expression_rows.append(read_rnaseq(sample_id))
my_expression_df = pd.concat(expression_rows)
my_expression_df.to_csv(os.path.join(out_dir, 'transcriptomic_features.csv'), index=False)

my_expression_df.head()

,sample_id,LASP1,HOXA11,CREBBP,ETV1,CCL26,CD79B,PAX7,BTK,BRCA1,...,NUTM2D,AC129492.1,DDX3X,FANCG,SSX2,ETV5,CEBPA,LSM14A,CUX1,AC124319.1
0,TCGA-AA-3561-01A,1.396537,0.859355,0.363925,-0.554161,-0.379824,-0.545659,0.312532,-0.532756,0.320832,...,-3.000000,-1.617132,0.796210,-0.029893,-3.00000,0.072156,1.003862,1.269823,0.728119,-0.590197
0,TCGA-EL-A3H2-11A,1.199193,-3.000000,1.004106,0.494157,-1.149837,-0.796687,-2.449990,-0.681050,-0.046047,...,-0.333901,-3.000000,1.421546,-0.933349,-1.93552,0.683707,0.264288,1.636809,1.091053,-0.820032
0,TCGA-EL-A3H2-01A,1.284624,-2.123623,0.975446,-0.385937,-1.585826,-0.172286,-2.432746,-0.251578,-0.326167,...,-0.459261,-1.688993,1.336900,-0.958924,-3.00000,0.608551,0.588353,1.347832,1.100819,-0.393994
0,TCGA-L5-A88S-01A,1.398796,-0.940586,0.916364,-0.195454,0.029628,-0.558926,-0.546275,-0.143277,0.119899,...,-1.008699,-1.348099,1.111280,-1.009175,-3.00000,0.209214,-0.015620,0.934234,0.925489,-0.846435
0,TCGA-N8-A56S-01A,1.337523,1.121986,0.892978,0.470908,-0.848976,-0.494248,0.269118,-0.413230,0.051759,...,-0.631155,-0.956411,1.336123,-0.461845,-3.00000,0.931428,0.821545,1.678032,0.872854,-0.517618
